# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlahimani/Flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions
*Look before deciding: distributions of your key fields. Note the heavy tails.*

I checked impressions_90d, ctr, and word_count using .describe().

Impressions_90d has a heavy tail: most pages get under 3,615 impressions, but the highest page got 517,715. The average (5,200) is much higher than the typical page (731), which means a few extreme pages are pulling the average up.

Ctr also has a heavy tail: half of all pages have CTR under 0.07, but the max is 100. This is likely caused by pages with very few impressions, where one click looks like a huge percentage. This is why CTR alone can't be trusted without also checking impressions.

Word_count is more normal: the average (3,108) and typical value (2,877) are close, so it isn't as skewed. Also, 7,699 pages are missing word_count, which matches what the data dictionary already told us to expect.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#key fields
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/peddikotlahimani/Flyrank-internship/main/data/raw/content_refresh_anonymized.csv")

print("Loaded! Number of rows:", len(df))
df[["impressions_90d", "ctr", "word_count"]].describe()


Loaded! Number of rows: 30000


,impressions_90d,ctr,word_count
count,30000.000000,30000.000000,22301.000000
mean,5200.366300,0.510733,3107.760325
std,16838.019547,3.279162,1452.382598
min,1.000000,0.000000,8.000000
25%,81.000000,0.000000,2413.000000
50%,731.000000,0.070000,2877.000000
75%,3615.250000,0.290000,3666.000000
max,517715.000000,100.000000,9546.000000


## 2. Signal test #1 / #2 / #3 (verdict each)
*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal Test #1: Do pages with more words get more clicks?
Mini-test: I compared the click rate of pages with lots of words vs pages with few words.
Result: Long pages = 0.32 click rate. Short pages = 0.88 click rate. Short pages did better.
Verdict: OPPOSITE

Signal Test #2: Do pages updated more recently get more clicks?
Mini-test: I compared the click rate of pages updated recently vs pages updated a long time ago.
Result: Recently updated pages = 0.73 click rate. Older pages = 0.26 click rate. Fresh pages did better.
Verdict: CONFIRMED

Signal Test #3: Does the type of article change how many clicks it gets?
Mini-test: I compared the click rate for three article types: feedly, keyword, and comparison.
Result: Feedly = 2.79. Keyword = 0.34. Comparison = 0.13. Feedly did way better than the others.
Verdict: CONFIRMED

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#Signal Test #1: Does longer content get better CTR?
# split pages into "long" and "short" based on the middle point (median)
median_wc = df["word_count"].median()
long_pages = df[df["word_count"] >= median_wc]
short_pages = df[df["word_count"] < median_wc]

print("Average CTR - long pages:", long_pages["ctr"].mean())
print("Average CTR - short pages:", short_pages["ctr"].mean())


Average CTR - long pages: 0.3173932926829268
Average CTR - short pages: 0.8801157054444345


In [ ]:
# This cell is for CODE (numbers, a query, a check).
#Signal Test #2: Does recently-updated content get better CTR?
median_update = df["days_since_last_update"].median()
recent_pages = df[df["days_since_last_update"] <= median_update]
older_pages = df[df["days_since_last_update"] > median_update]

print("Average CTR - recently updated:", recent_pages["ctr"].mean())
print("Average CTR - older pages:", older_pages["ctr"].mean())

Average CTR - recently updated: 0.7334217824278332
Average CTR - older pages: 0.2607563322484789


In [ ]:
# This cell is for CODE (numbers, a query, a check).
#Signal Test #3: Does content_type affect CTR?
df.groupby("content_type")["ctr"].mean()

,ctr
content_type,
comparison article,0.131205
feedly article,2.791274
keyword article,0.344766


## 3. The flag-linked test
*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The flag position_tier assumes: better ranking position = more clicks.Test showed that's true ,top_3 pages get almost 10x more clicks than deep pages, and the numbers go down step by step as position gets worse (with one small exception where striking was a bit higher than page_3_5).hence, data supports that rule's assumption with one minor exception.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/peddikotlahimani/Flyrank-internship/main/data/raw/content_refresh_anonymized.csv")

print("Loaded! Number of rows:", len(df))
df.groupby("position_tier")["ctr"].mean()


Loaded! Number of rows: 30000


,ctr
position_tier,
deep,0.150212
page_1,0.652467
page_3_5,0.222484
striking,0.323239
top_3,1.483611


## 4. What this means in practice
*Two or three sentences: what a content team should take from this.*

Basically, if you want more people to click on a page, the biggest thing that matters is where it ranks on Google. Pages in the top 3 spots get way more clicks than pages buried lower down, so that's worth fixing first. Keeping pages updated also seems to help — older pages get fewer clicks than ones updated recently. But writing longer articles doesn't seem to help on its own — in fact, shorter pages actually did better in this data, so more words isn't automatically a good thing.
1.improve ranking position
2.updation matters , since people prefer to stay in update

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [yes ] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes ] No client names, URLs, or private queries anywhere
- [yes ] My claims use careful words: observed, measured, directional, decision-support
- [yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.